# Rugby Rulebook Text Analysis

This notebook analyzes how the language of rugby's laws/rulebooks has changed
over time (1845-2026), and relates that change to the growth in the number of
international test matches played.

It covers:
- Fitting an exponential growth curve to cumulative test matches per year
- Word count / unique word count growth in the rulebooks over time
- Rank-frequency (Zipf) analysis of word usage per rulebook edition, fit to a
  Zipf-Mandelbrot distribution
- Tracking modal verbs (e.g. "may", "can", "might", "could") that grant
  discretion, as a share of all modal verbs used
- Growth in the number of laws over time

## Requirements

```bash
pip install nltk numpy matplotlib pandas scipy scikit-learn
```

## Data

This notebook expects a `data/` folder (next to the notebook) laid out as:

```
data/
├── rugby_test_matches_per_year_backfilled.csv
├── rugby_laws_per_year.csv
├── hierarchical_trees/
├── interdependencies/
└── text_files/
    └── cleaned_relavent_texts/
        ├── 1845/
        ├── 1874-1889/
        ├── 1894-1924/
        ├── 1929/
        ├── 1934-1949/
        ├── 1954-1959/
        ├── 1969-1974/
        ├── 1985-1995/
        ├── 1997-1998/
        ├── 2000-2002/
        ├── 2003-2016/  
        └── 2018-2026/
```

If your data lives somewhere else, edit `DATA_DIR` (and/or `TEXTS_DIR`) in the
**Configuration** cell below — nothing else needs to change. `interdependencies/` isn't used by this notebook; only `text_files/cleaned_relavent_texts/` is read.



## Imports

In [ ]:
import nltk
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import matplotlib.cm as cm
import matplotlib.colors as colors
import matplotlib as mpl
import scipy.optimize as opt
from sklearn.metrics import r2_score
import sys
import csv
import math

## One-time setup: NLTK data

`nltk.word_tokenize` and `nltk.pos_tag` (used later to tokenize the rulebook
text and tag modal verbs) need NLTK data packages that aren't bundled with
the library. Run this cell once per machine/environment — it's a no-op if the
data is already present. *(This cell is new; the original notebook assumed
this data was already downloaded.)*

In [ ]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

## Configuration

Update `DATA_DIR` if your data lives somewhere other than next to this
notebook. Everything else is derived from it.

In [ ]:
DATA_DIR = Path('data')

MATCHES_CSV_PATH = DATA_DIR / 'rugby_test_matches_per_year_backfilled.csv'
LAWS_CSV_PATH = DATA_DIR / 'rugby_laws_per_year.csv'
TEXTS_DIR = DATA_DIR / 'text_files' / 'cleaned_relavent_texts'

## Cumulative test matches per year

Loads the CSV of cumulative international test matches played, by year, into
a `year -> total matches` dictionary.

In [ ]:
csv_path = MATCHES_CSV_PATH

years_matches = {}

with open(csv_path, 'r') as csvfile: # adds element to dictionary of year: total matches
    reader = csv.DictReader(csvfile)
    for row in reader:
        years_matches[int(row['year'])] = int(row['n_matches_sum'])

Exponential growth model used to fit cumulative matches vs. year: `y = A * exp(B*x)`.

In [ ]:
def matches_year_func(x, A, B):
    x = np.asarray(x, dtype=float)
    y = A * np.exp(B * x)
    return y

Fit the exponential model to the year vs. cumulative-matches data.

In [ ]:
initial_guesses = [1, .01]

match_years = list(years_matches.keys())
#match_years = [year for year in match_years if years_matches[year] >= 100]
matches = list(years_matches.values())
#matches = [nmatch for nmatch in matches if nmatch >= 100]

bestfit_matches, bestfit_cov_matches = opt.curve_fit(matches_year_func, match_years, matches, p0=initial_guesses, maxfev=1000000)

Plot cumulative test matches vs. year (log y-axis), overlaid with the
best-fit exponential curve. The 1991 point (first women's test) is
highlighted in blue.

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(match_years, matches, color='black', s=5)
plt.scatter(1991, 1706, color='dodgerblue', s=20)
plt.plot(match_years, matches_year_func(match_years, bestfit_matches[0], bestfit_matches[1]), color='r')
plt.yscale('log')
plt.xticks(range(1880, 2030, 20))
plt.xlabel('Year', size=16)
plt.ylabel('Cumulative test matches', size=16)
plt.annotate('slope: ' + str(round(bestfit_matches[1],6)), (1880, 4000))
plt.annotate("first woman's", (1991, 1706), textcoords="offset points", xytext=(2, -10), ha='left', color='dodgerblue')
plt.annotate("test", (1991, 1706), textcoords="offset points", xytext=(2, -20), ha='left', color='dodgerblue')
print(plt.gca().get_xlim())

## Loading the rulebook texts

Reads every cleaned rulebook `.txt` file from each year-range subfolder under
`TEXTS_DIR`, and builds a single `year -> full text` dictionary. File names
are expected to look like `clean_<year>.txt`.

In [ ]:
def read_files_by_grouping():
    folder2003_path = TEXTS_DIR / '2003-2016'
    folder2018_path = TEXTS_DIR / '2018-2026'
    folder2000_path = TEXTS_DIR / '2000-2002'
    folder1997_path = TEXTS_DIR / '1997-1998'
    folder1985_path = TEXTS_DIR / '1985-1995'
    folder1969_path = TEXTS_DIR / '1969-1974'
    folder1954_path = TEXTS_DIR / '1954-1959'
    folder1934_path = TEXTS_DIR / '1934-1949'
    folder1929_path = TEXTS_DIR / '1929'
    folder1894_path = TEXTS_DIR / '1894-1924'
    folder1874_path = TEXTS_DIR / '1874-1889'
    folder1845_path = TEXTS_DIR / '1845'


    rulebooks = {} # empty dictionary to save data

    # cycle through all files
    for file_path in (folder2003_path.glob('*.txt')):
        with open(file_path, 'r') as file: # adds element to dictionary of file_name: file_text
            file_name = file_path.stem

            # same dictionary key as integer of just year
            year = file_name.replace('clean_', '')
            rulebooks[int(year)] = file.read()
    for file_path in (folder2018_path.glob('*.txt')):
        with open(file_path, 'r') as file: # adds element to dictionary of file_name: file_text
            file_name = file_path.stem

            # same dictionary key as integer of just year
            year = file_name.replace('clean_', '')
            rulebooks[int(year)] = file.read()
    for file_path in (folder2000_path.glob('*.txt')):
        with open(file_path, 'r') as file: # adds element to dictionary of file_name: file_text
            file_name = file_path.stem

            # same dictionary key as integer of just year
            year = file_name.replace('clean_', '')
            rulebooks[int(year)] = file.read()
    for file_path in (folder1997_path.glob('*.txt')):
        with open(file_path, 'r') as file: # adds element to dictionary of file_name: file_text
            file_name = file_path.stem

            # same dictionary key as integer of just year
            year = file_name.replace('clean_', '')
            rulebooks[int(year)] = file.read()
    for file_path in (folder1985_path.glob('*.txt')):
        with open(file_path, 'r') as file: # adds element to dictionary of file_name: file_text
            file_name = file_path.stem

            # same dictionary key as integer of just year
            year = file_name.replace('clean_', '')
            rulebooks[int(year)] = file.read()
    for file_path in (folder1969_path.glob('*.txt')):
        with open(file_path, 'r') as file: # adds element to dictionary of file_name: file_text
            file_name = file_path.stem

            # same dictionary key as integer of just year
            year = file_name.replace('clean_', '')
            rulebooks[int(year)] = file.read()
    for file_path in (folder1954_path.glob('*.txt')):
        with open(file_path, 'r') as file: # adds element to dictionary of file_name: file_text
            file_name = file_path.stem

            # same dictionary key as integer of just year
            year = file_name.replace('clean_', '')
            rulebooks[int(year)] = file.read()
    for file_path in (folder1934_path.glob('*.txt')):
        with open(file_path, 'r') as file: # adds element to dictionary of file_name: file_text
            file_name = file_path.stem

            # same dictionary key as integer of just year
            year = file_name.replace('clean_', '')
            rulebooks[int(year)] = file.read()
    for file_path in (folder1929_path.glob('*.txt')):
        with open(file_path, 'r') as file: # adds element to dictionary of file_name: file_text
            file_name = file_path.stem

            # same dictionary key as integer of just year
            year = file_name.replace('clean_', '')
            rulebooks[int(year)] = file.read()
    for file_path in (folder1894_path.glob('*.txt')):
        with open(file_path, 'r') as file: # adds element to dictionary of file_name: file_text
            file_name = file_path.stem

            # same dictionary key as integer of just year
            year = file_name.replace('clean_', '')
            rulebooks[int(year)] = file.read()
    for file_path in (folder1874_path.glob('*.txt')):
        with open(file_path, 'r') as file: # adds element to dictionary of file_name: file_text
            file_name = file_path.stem

            # same dictionary key as integer of just year
            year = file_name.replace('clean_', '')
            rulebooks[int(year)] = file.read()
    for file_path in (folder1845_path.glob('*.txt')):
        with open(file_path, 'r') as file: # adds element to dictionary of file_name: file_text
            file_name = file_path.stem

            # same dictionary key as integer of just year
            year = file_name.replace('clean_', '')
            rulebooks[int(year)] = file.read()
    
    return rulebooks

Actually load the rulebook texts into `rulebooks` (`year -> text`).

In [ ]:
rulebooks = read_files_by_grouping()

Tokenizes each rulebook's text (lowercased, alphanumeric tokens only),
and builds several summary dictionaries: token lists, word counts, and
unique word counts, each keyed by year, plus versions keyed by cumulative
match count for years from 1871 onward (when match records begin).

In [ ]:
def rulebooks_values(rulebooks):
    rulebooks_token = {}
    rulebooks_lengths = {}
    rulebooks_uniq_lengths = {}
    rulebooks_lengths_match = {}
    rulebooks_uniq_lengths_match = {}

    for book in rulebooks:
        # creating dictionary year: tokenized words
        rulebooks_token[book] = nltk.word_tokenize(rulebooks[book].lower())
        rulebooks_token[book] = [word for word in rulebooks_token[book] if word.isalnum()] # removes characters

        # creates dictionaries with year: # of (unique) words
        rulebooks_lengths[book] = len(rulebooks_token[book])
        rulebooks_uniq_lengths[book] = len(set(rulebooks_token[book]))

        # cutting out first 25 years where test matches not recorded
        if year >= 1871: 
            matches_temp = years_matches[year]

        # make dictionary cumulative matches: (unique) words
        rulebooks_lengths_match[matches_temp] = rulebooks_lengths[year]
        rulebooks_uniq_lengths_match[matches_temp] = rulebooks_uniq_lengths[year]

    return rulebooks_token, rulebooks_lengths, rulebooks_uniq_lengths, rulebooks_lengths_match, rulebooks_uniq_lengths_match

Run the tokenization/summary step defined above.

In [ ]:
rulebooks_token, rulebooks_lengths, rulebooks_uniq_lengths, rulebooks_lengths_match, rulebooks_uniq_lengths_match = rulebooks_values(rulebooks)

## Word counts vs. cumulative matches

Power-law model for word count vs. cumulative matches: `y = a * x^b`. Sorts
and converts the match/word-count dictionaries into arrays for curve
fitting.

In [ ]:

def words_match_func(x, a, b):
    y = a * (x ** b)
    return y
initial_guesses = [1, 1]

matches_fittable = np.array(sorted(list(rulebooks_lengths_match.keys())), dtype=float) # converts to fittable form
words_fittable = np.array(sorted(list(rulebooks_lengths_match.values())), dtype=float) # converts to fittable form
uniq_words_fittable = np.array(sorted(list(rulebooks_uniq_lengths_match.values())), dtype=float) # converts to fittable form

Fit the power-law model to total word counts and to unique word counts.

In [ ]:
bestfit_words, bestfit_cov_words = opt.curve_fit(words_match_func, matches_fittable, words_fittable, p0=initial_guesses, maxfev=100000)
bestfit_uwords, bestfit_cov_uwords = opt.curve_fit(words_match_func, matches_fittable, uniq_words_fittable, p0=initial_guesses, maxfev=100000)

Plot total word count vs. cumulative matches (log-log), with the fitted curve.

In [ ]:
# words by year figure
plt.figure(figsize=(5,5))
plt.scatter(matches_fittable, words_fittable, color='black')
plt.plot(matches_fittable, words_match_func(matches_fittable, bestfit_words[0], bestfit_words[1]), label='slope: ' + str(round(bestfit_words[1],3)), color='teal')
plt.yscale('log')
#plt.ylim(500, 50000)
plt.ylabel('# of words', size=16)
plt.xscale('log')
plt.xlabel('Cumulative matches', size=16)
plt.legend(fontsize=14)
#plt.annotate('slope: ' + str(round(bestfit_words[1],3)), (5, 28000), size=16)

Two ways of expressing the growth rate implied by the fitted exponent.

In [ ]:
alpha_growth=np.exp(np.log(2)/bestfit_words[1])
alpha_growth2=2**(1/bestfit_words[1])
print(alpha_growth)
print(alpha_growth2)

Plot unique word count vs. cumulative matches (log-log), with the fitted
curve. Three specific points (1889, 1945, 2018) are highlighted in blue.

In [ ]:
colors_uniq = ['dodgerblue' if int(nmatches)==59 or int(nmatches)==551 or int(nmatches)==4469 else 'black' for nmatches in matches_fittable]
# 59 in 1889, 551 in 1945, 4469 in 2018

# words by year figure
plt.figure(figsize=(5,5))
plt.scatter(matches_fittable, uniq_words_fittable, color=colors_uniq)
plt.plot(matches_fittable, words_match_func(matches_fittable, bestfit_uwords[0], bestfit_uwords[1]), color='r')
plt.yscale('log')
#plt.ylim(500, 50000)
plt.ylabel('# of unique words', size=16)
plt.xscale('log')
plt.xlabel('Cummulative matches', size=16)
plt.annotate('slope: ' + str(round(bestfit_uwords[1],4)), (5, 1900))

## Unique words vs. total words

Power-law model relating unique word count to total word count.

In [ ]:
def words_uwords_func(x, a, b):
    y = a * (x ** b) 
    return y

bestfit_words_uw, bestfit_cov_words_uw = opt.curve_fit(words_uwords_func, words_fittable, uniq_words_fittable, p0=[1, 0.5], maxfev=100000)

Plot unique word count vs. total word count (log-log), with the fitted curve.

In [ ]:
# # words by # unique words
plt.figure(figsize=(5,5))
plt.scatter(words_fittable, uniq_words_fittable, color='black')
plt.plot(words_fittable, words_uwords_func(words_fittable, bestfit_words_uw[0], bestfit_words_uw[1]), color='teal')
plt.yscale('log')
plt.ylabel('# of unique words', size=16)
plt.xscale('log')
plt.xlabel('# of words', size=16)
plt.annotate('slope: ' + str(round(bestfit_words_uw[1],4)), (2500, 1800))

## Rank-frequency distributions

For each year, computes a frequency distribution of tokens and converts it
into a `rank -> relative frequency` dictionary (rank 1 = most common word).

In [ ]:
freq_dists = {}

# returns dictionary of year: dictionary of rank: relative freq
for year in rulebooks_token:
    freq_dist = nltk.FreqDist(rulebooks_token[year]) # all words and counts

    rank_freq = {}
    for rank, (word, freq) in enumerate(freq_dist.most_common(), start=1):
        rank_freq[rank] = freq / rulebooks_lengths[year] # relative freq
    
    freq_dists[year] = rank_freq

Plot rank vs. relative frequency (log-log) for every year on one axis,
colored on a blue-red scale by year, with a colorbar.

In [ ]:
fig, ax = plt.subplots(figsize=(6,5))
ax.set_yscale('log')
ax.set_xscale('log')
ax.set_ylabel('Relative word frequency', size=16)
ax.set_xlabel('Rank', size=16)

# set color scale for years
norm = colors.Normalize(vmin=1845, vmax=2026)
colormap = plt.get_cmap('coolwarm')

# for each year, plots rank-freq points
for year in freq_dists:
    rank_freq_data = freq_dists[year]
    color = colormap(norm(year)) # color decided by year in smooth spectrum
    ax.scatter(rank_freq_data.keys(), rank_freq_data.values(), color=color, s=3)

# makes colorbar actually the right colors
sm = mpl.cm.ScalarMappable(cmap='coolwarm', norm=mpl.colors.Normalize(vmin=1845, vmax=2026))
sm.set_array([])

cbar = fig.colorbar(sm, ax=ax, ticks=[1850, 1900, 1950, 2000])
cbar.set_label("Year", fontsize=15)

plt.show()

## Zipf-Mandelbrot fits

Zipf-Mandelbrot model: `y = A / (r + beta)^alpha`.

In [ ]:
# defines function to fit to
def zipf_mandel_func(r, A, alpha, beta):
    y = A / ((r + beta) ** (alpha))
    return y


For each year, fits the Zipf-Mandelbrot model to that year's
rank-frequency data, records the fitted parameters and R², and plots the
data with the fitted curve (one small figure per year).

In [ ]:
zipf_mandel_values = {} 
zipf_mandel_exponents = {}
zipf_mandel_r2 = {}

initial_guesses = [0.1, 1, 0.5]

# for each year, fits zipf-mandelbrot function to rank-freq distribution
for year in freq_dists:
    rank_freq_data = freq_dists[year] # returns dictionary of rank: frequency
    ranks = np.array(list(rank_freq_data.keys()), dtype=float) # converts to fittable form
    freqs = np.array(list(rank_freq_data.values()), dtype=float)

    bestfit, bestfit_cov = opt.curve_fit(zipf_mandel_func, ranks, freqs, p0=initial_guesses, 
                                         bounds=([0, 0, 0], [np.inf, np.inf, np.inf]), maxfev=100000)
    
    # predicts values along best-fit function
    freq_pred = zipf_mandel_func(ranks, bestfit[0], bestfit[1], bestfit[2])
    rsquared = r2_score(freqs, freq_pred)
    
    # saves relavent values to dictionary
    zipf_mandel_values[year] = (bestfit, bestfit_cov)
    zipf_mandel_exponents[year] = bestfit[1]
    zipf_mandel_r2[year] = rsquared

    plt.figure(figsize=(3,3))
    plt.yscale('log')
    plt.xlabel('Rank')
    plt.ylabel('Relative frequency')
    plt.xscale('log')
    plt.title(year)
    plt.text(2400,0.1,'\u03B1 = ' + str(round(bestfit[1],3)), ha='right')
    plt.text(2400,0.05,'\u03B2 = ' + str(round(bestfit[2],8)), ha='right')
    plt.text(2400,0.025, 'R^2 = ' + str(round(rsquared,3)), ha='right')
    plt.scatter(ranks, freqs, s=2, color='black')
    plt.plot(ranks, freq_pred, 'r', label='Best fit')

Plot the fitted Zipf-Mandelbrot exponent (alpha) by year, highlighting
1998 and 2000 in red.

In [ ]:
# unique words by year
plt.figure(figsize=(5,5))
color = ['red' if (year == 2000 or year == 1998) else 'black' for year in zipf_mandel_exponents]
plt.scatter(zipf_mandel_exponents.keys(), zipf_mandel_exponents.values(), color=color)
plt.annotate(1998, (1998, zipf_mandel_exponents[1998]), textcoords="offset points", xytext=(25, -7), ha='center', size=16, color='red')
plt.annotate(2000, (2000, zipf_mandel_exponents[2000]), textcoords="offset points", xytext=(-25, -7), ha='center', size = 16, color='red')
plt.xlabel('Year', size=16)
plt.ylabel('Zipf-Mandelbrot exponent', size=16)

## Modal verb / discretionary language analysis

Quick check: POS-tag the 1845 rulebook and pull out all words tagged as
modal verbs (`MD`).

In [ ]:
# testing discretionary word filtering
tagged_example = nltk.pos_tag(rulebooks_token[1845])
modal_example = [word for (word, tag) in tagged_example if tag=='MD']
print(set(modal_example))

Do the same POS-tagging / modal-verb extraction for every year.

In [ ]:
tagged_words = {}
modal_verbs = {}
modal_verbs_sets = {}

for year in rulebooks_token:
    tagged_words[year] = nltk.pos_tag(rulebooks_token[year])
    modal_verbs[year] = [word for (word, tag) in tagged_words[year] if tag=='MD']
    modal_verbs_sets[year] = set(modal_verbs[year])

print(modal_verbs)
print(modal_verbs_sets)


Collect the full set of distinct modal verbs used across all years.

In [ ]:
modal_verbs_total_set = []

for year in modal_verbs_sets:
    modal_verbs_total_set += modal_verbs_sets[year]

print(set(modal_verbs_total_set))
    

Define which modal verbs are considered "discretion granting"
(`might`, `could`, `may`, `can`), and compute what percentage of each year's
modal verbs fall into that set.

In [ ]:
granting_modal_options = ['might', 'could', 'may', 'can']

granting_modals = {}
granting_modals_percents = {}

for year in modal_verbs: 
    granting_modals[year] = [word for word in modal_verbs[year] if word in granting_modal_options]
    granting_modals_percents[year] = len(granting_modals[year]) / len(modal_verbs[year])

Convert the discretion-granting percentages to be keyed by cumulative matches (from 1871 onward).

In [ ]:
matches_modal_percents = {}

# make dictionary cumulative matches: percent
for year in granting_modals_percents:
    if year >= 1871: # cutting out first 25 years where test matches not recorded
        matches_temp = years_matches[year]

        matches_modal_percents[matches_temp] = granting_modals_percents[year]

Plot the discretion-granting percentage vs. cumulative matches (log x-axis).

In [ ]:
# discretionary vocab percent out of all modal verbs
plt.figure(figsize=(5,5))
plt.scatter(matches_modal_percents.keys(), matches_modal_percents.values(), color='black')
plt.xlabel('Cummulative matches', size=16)
plt.ylabel('Discretion granting percent', size=16)
plt.xscale('log')

## Number of laws over time

Loads the CSV of the number of laws per year into a `year -> laws` dictionary.

In [ ]:
csv_laws_path = LAWS_CSV_PATH

years_laws = {}

with open(csv_laws_path, 'r') as csvfile: # adds element to dictionary of year: law no. 
    reader = csv.DictReader(csvfile)
    for row in reader:
        years_laws[int(row['\ufeffyear'])] = int(row['laws'])

Plot the number of laws vs. year.

In [ ]:
# words by year figure
plt.figure(figsize=(5,5))
plt.scatter(years_laws.keys(), years_laws.values(), color='black')
plt.ylabel('# of laws', size=16)
plt.xlabel('Year', size=16)

Convert the laws-per-year dictionary to be keyed by cumulative matches (from 1871 onward).

In [ ]:
matches_laws = {}

# make dictionary cumulative matches: words
for year in years_laws: 
    # cutting out first 25 years where test matches not recorded
    if year >= 1871: 
        matches_temp = years_matches[year]

    matches_laws[matches_temp] = years_laws[year]

Plot the number of laws vs. cumulative matches (log x-axis).

In [ ]:
# words by year figure
plt.figure(figsize=(5,5))
plt.scatter(matches_laws.keys(), matches_laws.values(), color='black')
plt.ylabel('# of laws', size=16)
plt.xscale('log')
plt.xlabel('Cummulative matches', size=16)